# PUL7SAR Phase 18 — Golden Editorial v6 / T4 Engineering Preview
This notebook separates durable image generation from semantic QA. Step 2 always generates, saves, verifies provenance, and displays Candidate 1 before any Qwen inspection can run. Native BF16 uses the Golden-reference precision path; a compatible legacy GPU may use the explicit FP16 engineering-preview path. Both preserve the PREVIEW composition without deterministic pitch replacement. Exact PUL7SAR branding and typography remain later deterministic layers. All execution remains `$0-local`; no paid image provider or hosted-GPU fallback is authorized.


## 1 — GPU compatibility preflight
Run this cell first. It requires at least 13 GB VRAM. If the GPU proves native BF16 it selects Golden Reference automatically; otherwise it selects the explicit T4 Engineering Preview in FP16. Nothing silently changes precision.


In [ ]:
import subprocess, torch
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout)
assert gpu.returncode == 0 and torch.cuda.is_available(), 'CUDA GPU runtime is required'
device = torch.cuda.current_device()
props = torch.cuda.get_device_properties(device)
vram_gb = props.total_memory / (1024 ** 3)
bf16_ok = bool(getattr(torch.cuda, 'is_bf16_supported', lambda: False)())
assert vram_gb >= 13.0, f'PUL7SAR FLUX.2 execution requires at least 13 GB VRAM; assigned GPU has {vram_gb:.2f} GB'
precision_mode = 'auto' if bf16_ok else 'float16-preview'
precision_label = 'GOLDEN REFERENCE — BF16' if bf16_ok else 'T4 ENGINEERING PREVIEW — FP16 (NOT GOLDEN)'
print(f'GPU: {props.name} | VRAM: {vram_gb:.2f} GB | native BF16: {bf16_ok}')
print(f'PUL7SAR MODE: {precision_label}')


## 2 — Generate, save and display Candidate 1
Run this cell only after step 1 succeeds. It clones the protected Phase 18 branch, installs the optional GPU dependencies, runs the full Phase 18 CPU validation, then calls `phase18_colab_runner.py` directly. The runner writes the native image, exact-canvas PNG, durable result JSON, provenance metadata, and `latest.json` before this cell displays the PNG. Qwen is deliberately not loaded in this step, so a later semantic-QA problem cannot hide or destroy a successfully generated Candidate 1. The story-first v6 composition remains locked: illuminated tunnel lower-left, right-center copy space, upper-left brand quiet zone, context-only turf, and no deterministic football-pitch replacement.


In [ ]:
import json, os, shutil, subprocess, sys
from pathlib import Path
from IPython.display import Image, display
assert 'precision_mode' in globals(), 'Run GPU compatibility preflight first'
repo = '/content/pul7sar-bot'
if os.path.isdir(repo):
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '--branch', 'phase18/story-intelligence', 'https://github.com/pulsar7official/pul7sar-bot.git', repo], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-r', f'{repo}/requirements-phase18-gpu.txt'], check=True)
print('Running Phase 18 CPU validation before GPU generation...', flush=True)
cpu = subprocess.run([sys.executable, '-u', f'{repo}/tools/phase18_cpu_validate.py'], cwd=repo)
if cpu.returncode != 0:
    raise RuntimeError(f'PUL7SAR CPU validation failed with exit code {cpu.returncode}; GPU generation blocked.')
command = [sys.executable, '-u', f'{repo}/tools/phase18_colab_runner.py', '--candidate', '1', '--dtype', precision_mode, '--skip-targeted-tests', '--force']
print('Executing generation-only path:', ' '.join(command), flush=True)
run_env = os.environ.copy()
run_env['PYTHONUNBUFFERED'] = '1'
result = subprocess.run(command, cwd=repo, env=run_env)
if result.returncode != 0:
    raise RuntimeError(f'PUL7SAR Candidate 1 generation failed with exit code {result.returncode}; inspect the traceback immediately above.')
latest_path = Path(repo) / 'output' / 'phase18_colab' / 'latest.json'
assert latest_path.is_file(), 'Candidate 1 completed but latest.json is missing'
latest = json.loads(latest_path.read_text(encoding='utf-8'))
png_path = Path(str(latest.get('png', '')))
if not png_path.is_absolute():
    png_path = Path(repo) / png_path
assert png_path.is_file(), f'Candidate 1 PNG is missing: {png_path}'
candidate1_png = str(png_path.resolve())
print('\n=== CANDIDATE 1 SAVED SUCCESSFULLY ===')
print('PNG:', candidate1_png)
print('Status:', latest.get('status'))
print('Resolved dtype:', latest.get('resolved_dtype'))
print('Semantic QA has NOT run yet; publication_ready remains False.')
display(Image(filename=candidate1_png))


## 3 — Optional semantic QA after you have reviewed the pixels
Do not run this cell until Candidate 1 is visibly displayed and manually reviewed. This is intentionally a separate process. It may load Qwen and can be memory/network intensive, but it can no longer prevent Step 2 from saving and showing the FLUX image. Even a successful semantic result does not make the image publication-ready; exact branding, typography, and later publication gates remain separate.


In [ ]:
import subprocess, sys
assert 'candidate1_png' in globals(), 'Run Step 2 and display Candidate 1 first'
semantic_command = [sys.executable, '-u', f'{repo}/tools/phase18_colab_one_command.py', '--candidate', '1', '--semantic-inspection', 'qwen', '--skip-update']
print('Semantic QA is optional at this stage. Executing:', ' '.join(semantic_command), flush=True)
semantic_result = subprocess.run(semantic_command, cwd=repo)
if semantic_result.returncode != 0:
    print('Semantic QA did not complete. Candidate 1 PNG remains safely saved at:', candidate1_png)
else:
    print('Semantic QA completed. Candidate 1 remains publication_ready=False until later gates.')


## Quality acceptance rule
A generated PNG is not automatically accepted. A BF16 result may continue through the Golden-reference gates. A T4 FP16 result is only an engineering preview and cannot satisfy the Golden precision gate or become publication-ready. We first judge focal hierarchy, depth, atmosphere, right-center copy space, upper-left brand quiet zone, realism, and absence of pitch-template dominance from the saved pixels themselves. The strict Golden floor remains 8.5; 9.0+ is the elite target. Exact branding and typography are added only after the base image survives visual review. Semantic/runtime/integrity failures remain fail-closed for publication but can no longer erase or hide an already generated engineering proof.
